# CDD-11 discovery: 2 models × 11 degradations × 78 scenes
This run continues the successful smoke test. It evaluates only the locked discovery partition from the CDD-11 test archive. No training, confirmation or sealed holdout. Internet + GPU T4 required. Save Version → Save & Run All. Download `common_failure_discovery_results.zip` after completion, including when a model fails. The notebook downloads the 3.77 GB source archive again in a fresh Kaggle version. Estimated runtime is provisional; do not assume the smoke timing is an end-to-end benchmark.


In [ ]:
import os, sys, subprocess, json, shutil, traceback
from pathlib import Path
REPO = Path('/kaggle/working/CoT-restoration')
URL = 'https://github.com/HoangKhanhTung0111/CoT-restoration.git'
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','--depth','1',URL,str(REPO)],check=True)
os.chdir(REPO)
WORK = Path('/kaggle/working/common_failure_discovery')
WORK.mkdir(exist_ok=True)
print('Project revision:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','huggingface_hub','gdown','einops','timm','fvcore','thop','scikit-image'],check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running.'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
import os, sys, subprocess, json, shutil, traceback
from pathlib import Path
REPO = Path('/kaggle/working/CoT-restoration')
WORK = Path('/kaggle/working/common_failure_discovery')
BUNDLE = WORK / 'bundle'
BUNDLE.mkdir(parents=True, exist_ok=True)
errors = []
def run_logged(args, name):
    with (BUNDLE / (name+'.log')).open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','-m','hybrid_cot_nafnet.common_failure_audit',*args,'--work',str(WORK)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end=''); log.write(line); log.flush()
        if process.wait() != 0: raise RuntimeError(name+' failed; send the result ZIP')
try:
    run_logged(['prepare'],'prepare')
    for model in ['onerestore','mirage']:
        try: run_logged(['audit','--model',model],model)
        except Exception as exc: errors.append(str(exc))
except Exception: errors.append(traceback.format_exc())
finally:
    for name in ['manifest.json','sources.json']:
        if (WORK/name).exists(): shutil.copy2(WORK/name,BUNDLE/name)
    if (WORK/'discovery').exists(): shutil.copytree(WORK/'discovery',BUNDLE/'discovery',dirs_exist_ok=True)
    (BUNDLE/'run.json').write_text(json.dumps({'errors':errors,'project_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()},indent=2))
    (BUNDLE/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    archive = shutil.make_archive('/kaggle/working/common_failure_discovery_results','zip',BUNDLE)
    print('Download:',archive)
if errors: raise RuntimeError('Discovery incomplete. Send ZIP: '+str(errors))
print('Discovery inference complete. Failure-mode analysis requires paired metrics and image review.')
